# 02 — Validation outputs (Sub-sesión 1b)

Carga `_grid_summary.parquet` generado por `scripts/generate_all_masters.py` y produce
análisis + gráficos. **No regenera datos.** Pre-requisito: haber ejecutado
`python -m scripts.generate_all_masters` desde `validation_study/`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scripts import config as cfg
from scripts import grid_utils

pd.set_option('display.max_columns', 100)
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Carga del resumen

In [ ]:
summary_path = cfg.STUDY_MASTERS / '_grid_summary.parquet'
assert summary_path.exists(), f'No encontrado: {summary_path}. Ejecuta scripts/generate_all_masters.py primero.'

df = pd.read_parquet(summary_path)
print(f'Configs procesadas: {len(df)}')
print(f'Masters totales:    {len(df) * 3}')
df.head(10)

In [ ]:
# Tabla descriptiva
df.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(2)

## 2. N usuarios por config

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df_sorted = df.sort_values('cutoff')
ax.bar(df_sorted['config_id'], df_sorted['n_users'])
ax.set_xticks(range(len(df_sorted)))
ax.set_xticklabels(df_sorted['config_id'], rotation=90, fontsize=8)
ax.axhline(25380, color='red', linestyle='--', label='Baseline L32 (25,380)')
ax.set_ylabel('N usuarios')
ax.set_title('Tamaño del sample por config')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Heatmap: nº features tras cleanup × cleanup_version

In [ ]:
heat = df.set_index('config_id')[['n_features_base', 'n_features_v1', 'n_features_v2', 'n_features_v3']]
fig, ax = plt.subplots(figsize=(8, 14))
sns.heatmap(heat, annot=True, fmt='d', cmap='YlGnBu', cbar_kws={'label': 'n cols'}, ax=ax)
ax.set_title('Nº features por config × cleanup')
plt.tight_layout()
plt.show()

## 4. Drops por categoría

In [ ]:
drops_cols = ['n_dropped_high_nan', 'n_dropped_quasi_const',
              'n_dropped_target_leakage', 'n_dropped_corr_99', 'n_dropped_corr_95']
fig, ax = plt.subplots(figsize=(14, 5))
df_drops = df.set_index('config_id')[drops_cols].sort_index()
df_drops.plot(kind='bar', stacked=True, ax=ax)
ax.set_ylabel('Cols dropeadas')
ax.set_title('Drops por categoría y config')
ax.legend(loc='upper right', fontsize=8)
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.show()

## 5. Tiempos de cómputo

In [ ]:
df['time_total_s'] = df['time_master_s'] + df['time_cleanup_s']
print(f'Tiempo total acumulado: {df["time_total_s"].sum()/60:.1f} min')
print(f'Tiempo medio por config: {df["time_total_s"].mean():.1f}s')
print(f'Más lento: {df.nlargest(3, "time_total_s")[["config_id", "time_total_s"]].values.tolist()}')

fig, ax = plt.subplots(figsize=(14, 4))
df_sorted = df.sort_values('config_id')
ax.bar(df_sorted['config_id'], df_sorted['time_master_s'], label='build_master', alpha=0.7)
ax.bar(df_sorted['config_id'], df_sorted['time_cleanup_s'], bottom=df_sorted['time_master_s'], label='cleanup', alpha=0.7)
ax.set_ylabel('Segundos')
ax.set_title('Tiempo de build_master + cleanup por config')
ax.legend()
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.show()

## 6. Generar informe ejecutivo

In [ ]:
report_path = cfg.STUDY_INFORMES / '02b_masters_report.md'

report = f'''# Masters generados — Sub-sesion 1b

## Resumen

- Configs procesadas: {len(df)} / 45 esperados
- Masters totales: {len(df) * 3}
- Tiempo total de cómputo: {df["time_total_s"].sum()/60:.1f} min

## Tamaño del sample

| Stat | N users |
|---|---:|
| min | {df["n_users"].min():,} |
| mediana | {int(df["n_users"].median()):,} |
| max | {df["n_users"].max():,} |

## Features por cleanup (mediana)

| Versión | mediana cols |
|---|---:|
| base | {int(df["n_features_base"].median())} |
| v1_conservative | {int(df["n_features_v1"].median())} |
| v2_intermediate | {int(df["n_features_v2"].median())} |
| v3_aggressive | {int(df["n_features_v3"].median())} |

## Próximo paso

Fase 3b-2: screening con LightGBM/XGBoost/CatBoost sobre las 3 versiones de cada config.
Modelos totales: 45 configs × 3 cleanups × 3 algoritmos × N targets (1 o 2 según config) ≈ 729.
'''

report_path.write_text(report)
print(f'Reporte escrito: {report_path}')